# Signal Amplitude And Phase Visualization

Purpose: inspect CSI phase and amplitude across streams, time, and subcarriers with a deterministic synthetic capture.

Run path: install research extras with `uv sync --extra research`, open this notebook, and choose Run All. The notebook is self-contained and does not require hardware recordings.

Fixture / simulated source: `make_phase_amplitude_fixture` creates a 160-frame, 3-stream, 64-subcarrier CSI capture with smooth static channel structure, breathing-like low-rate phase movement, a localized motion burst, and tiny seeded noise.

Plots: stream-0 amplitude heatmap, stream-0 wrapped phase heatmap, center-subcarrier unwrapped phase by stream, and center-band amplitude over time.


In [ ]:
# Data source option: synthetic fixture or local ESP32 CSI recording.
USE_RECORDING = False
RECORDING_CSV = None  # Set to a specific *_csi.csv path, or leave None to auto-pick from data/recordings.
_RECORDING_MAX_FRAMES = None

from pathlib import Path
import numpy as np

try:
    from ruview.hardware.esp32_capture_analysis import load_esp32_capture
except ImportError:  # Allows the notebook to render in environments without the package installed yet.
    load_esp32_capture = None


def _find_repo_root():
    current = Path.cwd().resolve()
    for candidate in (current, *current.parents):
        if (candidate / 'pyproject.toml').exists() and (candidate / 'src' / 'ruview').exists():
            return candidate
    return current


def _pick_recording_csv(repo_root, override=None):
    if override:
        return Path(override).expanduser().resolve()
    candidates = sorted((repo_root / 'data' / 'recordings').glob('**/*_csi.csv'))
    return candidates[0] if candidates else None


def _elapsed_seconds_for_capture(capture):
    real_ts = np.asarray(capture.timestamps, dtype=np.float64)
    if np.isfinite(real_ts).sum() >= 2 and float(np.nanmax(real_ts) - np.nanmin(real_ts)) > 0:
        return real_ts - float(np.nanmin(real_ts))
    mono = np.asarray(capture.host_monotonic_ns, dtype=np.float64)
    if mono.size == 0:
        return np.array([], dtype=np.float64)
    return (mono - float(mono[0])) / 1_000_000_000.0


def _fill_invalid(values, mask):
    filled = np.asarray(values, dtype=np.float64).copy()
    filled[~mask] = np.nan
    valid = np.isfinite(filled)
    counts = valid.sum(axis=0)
    sums = np.where(valid, filled, 0.0).sum(axis=0)
    col_means = np.divide(sums, counts, out=np.zeros_like(sums), where=counts > 0)
    rows, cols = np.where(~valid)
    filled[rows, cols] = col_means[cols]
    return filled


def _capture_to_csi_tensor(capture, max_frames=None):
    take = slice(None) if max_frames is None else slice(0, max_frames)
    amp = _fill_invalid(capture.amplitude[take], capture.valid_mask[take])
    phase = _fill_invalid(capture.phase[take], capture.valid_mask[take])
    time_s = _elapsed_seconds_for_capture(capture)[take]
    csi = (amp * np.exp(1j * phase))[:, None, :]
    return csi, time_s


def _effective_sample_rate(time_s):
    time_s = np.asarray(time_s, dtype=np.float64)
    duration = float(time_s[-1] - time_s[0]) if time_s.size >= 2 else 0.0
    return float((time_s.size - 1) / duration) if duration > 0 else 1.0


_recording_repo_root = _find_repo_root()
_recording_path = _pick_recording_csv(_recording_repo_root, RECORDING_CSV)
_recording_capture = None
_recording_csi = None
_recording_time_s = None

if USE_RECORDING:
    if load_esp32_capture is None:
        raise ImportError('ruview.hardware.esp32_capture_analysis.load_esp32_capture is required for recordings')
    if _recording_path is None:
        raise FileNotFoundError('No *_csi.csv recording found under data/recordings; set RECORDING_CSV explicitly.')
    _recording_capture = load_esp32_capture(_recording_path)
    _recording_csi, _recording_time_s = _capture_to_csi_tensor(_recording_capture, _RECORDING_MAX_FRAMES)
    print(f'Using recording: {_recording_path} ({_recording_csi.shape[0]} frames, {_recording_csi.shape[2]} subcarriers)')
else:
    print('Using synthetic fixture. Set USE_RECORDING=True to plot from a local *_csi.csv recording.')


import numpy as np

try:
    import matplotlib.pyplot as plt
except Exception as exc:
    raise RuntimeError('Install the research extra with `uv sync --extra research` to run this notebook.') from exc


def make_phase_amplitude_fixture(
    n_frames=160,
    n_streams=3,
    n_subcarriers=64,
    sample_rate_hz=20.0,
    seed=404,
):
    rng = np.random.default_rng(seed)
    time_s = np.arange(n_frames, dtype=float) / sample_rate_hz
    streams = np.arange(n_streams, dtype=float)
    subcarriers = np.arange(n_subcarriers, dtype=float)

    center = 39.0
    body_band = np.exp(-0.5 * ((subcarriers - center) / 5.5) ** 2)
    static_amplitude = 1.0 + 0.07 * np.cos((subcarriers - 20.0) / 8.0)
    stream_gain = (1.0 + 0.045 * streams)[None, :, None]

    breathing = np.sin(2.0 * np.pi * 0.28 * time_s)[:, None, None]
    motion_burst = np.exp(-0.5 * ((time_s - 4.2) / 0.55) ** 2)[:, None, None]
    amplitude = stream_gain * (
        static_amplitude[None, None, :]
        + 0.08 * breathing * body_band[None, None, :]
        + 0.18 * motion_burst * body_band[None, None, :]
    )
    amplitude = amplitude + rng.normal(0.0, 0.008, size=amplitude.shape)

    static_phase = 0.05 * subcarriers[None, None, :] + 0.38 * streams[None, :, None]
    slow_drift = 0.035 * time_s[:, None, None]
    body_phase = 0.16 * np.sin(
        2.0 * np.pi * 0.28 * time_s[:, None, None] + 0.25 * streams[None, :, None]
    ) * body_band[None, None, :]
    wrapped_phase = np.angle(np.exp(1j * (static_phase + slow_drift + body_phase)))

    csi = amplitude * np.exp(1j * wrapped_phase)
    return {
        'csi': csi,
        'time_s': time_s,
        'subcarriers': subcarriers,
        'sample_rate_hz': sample_rate_hz,
        'body_band': body_band,
        'motion_window_s': (3.4, 5.0),
    }


if USE_RECORDING and _recording_csi is not None:
    csi = _recording_csi
    rec_time = _recording_time_s - float(_recording_time_s[0]) if _recording_time_s.size else _recording_time_s
    variance_band = np.nanvar(np.abs(csi[:, 0, :]), axis=0)
    max_variance = float(np.nanmax(variance_band)) if variance_band.size else 0.0
    body_band = variance_band / max(max_variance, 1e-12)
    fixture = {
        'csi': csi,
        'time_s': rec_time,
        'subcarriers': np.arange(csi.shape[2], dtype=float),
        'sample_rate_hz': _effective_sample_rate(rec_time),
        'body_band': body_band,
        'motion_window_s': (float(rec_time[0]), float(rec_time[-1])) if rec_time.size else (0.0, 0.0),
    }
else:
    fixture = make_phase_amplitude_fixture()
    csi = fixture['csi']
amplitude = np.abs(csi)
wrapped_phase = np.angle(csi)
unwrapped_phase = np.unwrap(wrapped_phase, axis=0)
relative_phase = unwrapped_phase - unwrapped_phase[0:1]

print(f"CSI shape [time, stream, subcarrier]: {csi.shape}")
print(f"Mean amplitude: {amplitude.mean():.3f}")
print(f"Wrapped phase range: {wrapped_phase.min():.2f} to {wrapped_phase.max():.2f} rad")


In [ ]:
time_s = fixture['time_s']
subcarriers = fixture['subcarriers']
center_index = int(np.argmax(fixture['body_band']))
extent = [subcarriers[0], subcarriers[-1], time_s[0], time_s[-1]]

fig, axes = plt.subplots(2, 2, figsize=(14, 8), constrained_layout=True)

amp_image = axes[0, 0].imshow(
    amplitude[:, 0, :],
    aspect='auto',
    origin='lower',
    extent=extent,
    cmap='viridis',
)
axes[0, 0].set_title('stream 0 amplitude')
axes[0, 0].set_xlabel('subcarrier')
axes[0, 0].set_ylabel('time [s]')
fig.colorbar(amp_image, ax=axes[0, 0], label='amplitude')

phase_image = axes[0, 1].imshow(
    wrapped_phase[:, 0, :],
    aspect='auto',
    origin='lower',
    extent=extent,
    cmap='twilight',
    vmin=-np.pi,
    vmax=np.pi,
)
axes[0, 1].set_title('stream 0 wrapped phase')
axes[0, 1].set_xlabel('subcarrier')
axes[0, 1].set_ylabel('time [s]')
fig.colorbar(phase_image, ax=axes[0, 1], label='phase [rad]')

for stream_index in range(csi.shape[1]):
    axes[1, 0].plot(
        time_s,
        relative_phase[:, stream_index, center_index],
        label=f'stream {stream_index}',
    )
axes[1, 0].axvspan(*fixture['motion_window_s'], color='tab:orange', alpha=0.14, label='motion burst')
axes[1, 0].set_title(f'unwrapped phase at subcarrier {center_index}')
axes[1, 0].set_xlabel('time [s]')
axes[1, 0].set_ylabel('relative phase [rad]')
axes[1, 0].legend(loc='upper left')

band_start = max(0, center_index - 3)
band_stop = min(csi.shape[2], center_index + 4)
center_band = slice(band_start, band_stop)
for stream_index in range(csi.shape[1]):
    axes[1, 1].plot(
        time_s,
        amplitude[:, stream_index, center_band].mean(axis=1),
        label=f'stream {stream_index}',
    )
axes[1, 1].axvspan(*fixture['motion_window_s'], color='tab:orange', alpha=0.14, label='motion burst')
axes[1, 1].set_title('center-band amplitude over time')
axes[1, 1].set_xlabel('time [s]')
axes[1, 1].set_ylabel('mean amplitude')
axes[1, 1].legend(loc='upper left')

plt.show()


Expected interpretation: the amplitude heatmap should show a stable static channel plus a localized high-energy band near the synthetic body subcarriers. Wrapped phase should stay bounded in `[-pi, pi]`, while the unwrapped center-subcarrier trace should reveal smooth low-rate phase motion and a visible drift shared across streams.

Limitations: this is a deterministic visual fixture, not a calibrated RF capture. It does not model packet loss, CFO/SFO correction, antenna coupling, AGC changes, multi-person interference, ESP32 quantization, or phase-sanitizer parity with the Rust implementation. Use it as a shape and plotting smoke test before replacing the fixture with recorded CSI.
